# Emotional Monitoring

### Load Libraries

In [27]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

In [34]:
# load dataset
df = pd.read_csv('../../Datasets/emotional_monitoring/emotional_monitoring_dataset_with_target.csv')
df.head()

,HeartRate,SkinConductance,EEG,Temperature,PupilDiameter,SmileIntensity,FrownIntensity,CortisolLevel,ActivityLevel,AmbientNoiseLevel,LightingLevel,EmotionalState,CognitiveState,EngagementLevel
0,61,8.937204,11.794946,36.501723,3.330181,0.689238,0.189024,0.603035,136,59,394,engaged,distracted,3
1,60,12.635397,19.151412,36.618910,3.428995,0.561056,0.091367,0.566671,155,39,479,engaged,distracted,1
2,81,3.660028,6.226098,36.176898,2.819286,0.417951,0.227355,1.422475,55,30,832,partially engaged,focused,3
3,119,0.563070,4.542968,37.205293,2.192961,0.140186,0.502965,1.669045,39,40,602,disengaged,focused,3
4,118,0.477378,0.996209,37.248118,2.450139,0.064471,0.695604,1.854076,10,42,908,disengaged,focused,3


In [29]:
# Encode categorical features
le_emotionalState = LabelEncoder()
le_cognitiveState = LabelEncoder()
df['EmotionalState'] = le_emotionalState.fit_transform(df['EmotionalState'])
df['CognitiveState'] = le_cognitiveState.fit_transform(df['CognitiveState'])

In [30]:
# Seprate features and target
X = df.drop(columns=['EngagementLevel'])
y = df['EngagementLevel']

# Adjust labels to start from 0 for PyTorch
y = y - y.min()
num_classes = len(y.unique())

# Splite data
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

In [31]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Dataset shape: {X.shape}')
print(f'# classes: {num_classes}')
print(f'Class distribution: {np.bincount(y)}\n')

Dataset shape: (1000, 13)
# classes: 3
Class distribution: [ 97 324 579]



In [35]:
# Encode categorical features
le_emotional = LabelEncoder()
le_cognitive = LabelEncoder()
df['EmotionalState'] = le_emotional.fit_transform(df['EmotionalState'])
df['CognitiveState'] = le_cognitive.fit_transform(df['CognitiveState'])

# Separate features and target
X = df.drop('EngagementLevel', axis=1).values
y = df['EngagementLevel'].values

# Adjust labels to start from 0 for PyTorch
y = y - y.min()
num_classes = len(np.unique(y))

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {num_classes}")
print(f"Class distribution: {np.bincount(y)}\n")

Dataset shape: (1000, 13)
Number of classes: 3
Class distribution: [ 97 324 579]



In [37]:
# ===== CLASSICAL ML MODELS =====

print("="*50)
print("CLASSICAL ML MODELS")
print("="*50)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
print(f"\nRandom Forest Accuracy: {accuracy_score(y_test, rf_pred):.4f}")

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train_scaled, y_train)
gb_pred = gb.predict(X_test_scaled)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_test, gb_pred):.4f}")

# SVM
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)
print(f"SVM Accuracy: {accuracy_score(y_test, svm_pred):.4f}")


CLASSICAL ML MODELS

Random Forest Accuracy: 1.0000
Gradient Boosting Accuracy: 1.0000
SVM Accuracy: 0.9300


In [36]:
# ===== PYTORCH DEEP LEARNING MODELS =====

class EmotionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create datasets
train_dataset = EmotionDataset(X_train_scaled, y_train)
test_dataset = EmotionDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

input_dim = X_train.shape[1]


In [23]:
# Model 1: Simple MLP
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(SimpleMLP, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.fc(x)

# Model 2: Deep MLP
class DeepMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(DeepMLP, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x):
        return self.fc(x)

# Model 3: Residual MLP
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super(ResidualBlock, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()
    
    def forward(self, x):
        return self.relu(x + self.fc(x))

class ResidualMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(ResidualMLP, self).__init__()
        self.input_layer = nn.Linear(input_dim, 64)
        self.res_block1 = ResidualBlock(64)
        self.res_block2 = ResidualBlock(64)
        self.output_layer = nn.Linear(64, num_classes)
    
    def forward(self, x):
        x = self.input_layer(x)
        x = self.res_block1(x)
        x = self.res_block2(x)
        return self.output_layer(x)


In [24]:
# Training function
def train_model(model, train_loader, test_loader, epochs=50, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            model.eval()
            correct = 0
            total = 0
            with torch.no_grad():
                for X_batch, y_batch in test_loader:
                    outputs = model(X_batch)
                    _, predicted = torch.max(outputs, 1)
                    total += y_batch.size(0)
                    correct += (predicted == y_batch).sum().item()
            
            accuracy = correct / total
            print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss/len(train_loader):.4f}, Test Acc: {accuracy:.4f}")
    
    return model

# Evaluation function
def evaluate_model(model, test_loader):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.numpy())
            all_labels.extend(y_batch.numpy())
    
    accuracy = accuracy_score(all_labels, all_preds)
    return accuracy, all_preds, all_labels


In [39]:
# Train all models
print("\n" + "="*50)
print("DEEP LEARNING MODELS")
print("="*50)

# input_dim = X_train_scaled.shape[1]

models = {
    'SimpleMLP': SimpleMLP(input_dim, num_classes),
    'DeepMLP': DeepMLP(input_dim, num_classes),
    'ResidualMLP': ResidualMLP(input_dim, num_classes)
}

results = {}

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    trained_model = train_model(model, train_loader, test_loader, epochs=150, lr=0.001)
    accuracy, preds, labels = evaluate_model(trained_model, test_loader)
    results[name] = accuracy
    print(f"\nFinal {name} Test Accuracy: {accuracy:.4f}")

# Summary
print("\n" + "="*50)
print("FINAL RESULTS SUMMARY")
print("="*50)
print(f"Random Forest: {accuracy_score(y_test, rf_pred):.4f}")
print(f"Gradient Boosting: {accuracy_score(y_test, gb_pred):.4f}")
print(f"SVM: {accuracy_score(y_test, svm_pred):.4f}")
for name, acc in results.items():
    print(f"{name}: {acc:.4f}")



DEEP LEARNING MODELS

--- Training SimpleMLP ---
Epoch 10/150, Loss: 0.5445, Test Acc: 0.7650
Epoch 20/150, Loss: 0.3758, Test Acc: 0.8900
Epoch 30/150, Loss: 0.2607, Test Acc: 0.9000
Epoch 40/150, Loss: 0.2133, Test Acc: 0.9350
Epoch 50/150, Loss: 0.1888, Test Acc: 0.9350
Epoch 60/150, Loss: 0.1605, Test Acc: 0.9550
Epoch 70/150, Loss: 0.1476, Test Acc: 0.9400
Epoch 80/150, Loss: 0.1148, Test Acc: 0.9500
Epoch 90/150, Loss: 0.1040, Test Acc: 0.9500
Epoch 100/150, Loss: 0.1167, Test Acc: 0.9450
Epoch 110/150, Loss: 0.0756, Test Acc: 0.9350
Epoch 120/150, Loss: 0.0861, Test Acc: 0.9450
Epoch 130/150, Loss: 0.0740, Test Acc: 0.9600
Epoch 140/150, Loss: 0.0959, Test Acc: 0.9500
Epoch 150/150, Loss: 0.0603, Test Acc: 0.9450

Final SimpleMLP Test Accuracy: 0.9450

--- Training DeepMLP ---
Epoch 10/150, Loss: 0.4877, Test Acc: 0.8900
Epoch 20/150, Loss: 0.3156, Test Acc: 0.9100
Epoch 30/150, Loss: 0.2759, Test Acc: 0.9300
Epoch 40/150, Loss: 0.2418, Test Acc: 0.9050
Epoch 50/150, Loss: 0.22